# 🧬 Dark Manifold V10 — Biologically-Correct Knockouts

## The V9 Problem

V9 used a **dense Linear layer** for flux gating:
```python
flux_gate = sigmoid(Linear(gene_expr))  # Every gene affects every reaction!
```

This dilutes knockout effects to ~0.15% — not enough to cause lethality.

## The V10 Fix

Use the **G matrix directly** — if a gene is required for a reaction, knocking it out STOPS that reaction:

```python
# G[gene, rxn] = 1 if gene is required for reaction
# flux_gate[rxn] = min(gene_expr[required genes])
# If ANY required gene = 0, flux_gate = 0
```

This is how **real enzymes work** — missing one subunit = no enzyme.

## Expected Results

- 123 reactions have 1 required gene → knockout stops reaction
- If reaction is essential → **lethal knockout**
- Expected: **50-70% essential genes**

In [ ]:
#@title 1️⃣ Setup
from google.colab import files
import pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import re
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

print("\nUpload syn3a_real_data.pkl:")
uploaded = files.upload()

with open('syn3a_real_data.pkl', 'rb') as f:
    raw_data = pickle.load(f)

print("✅ Data loaded!")

In [ ]:
#@title 2️⃣ Parse Data

dfs = raw_data['all_dataframes']
rxns_df = pd.DataFrame(dfs['reconstruction.xlsx:Reactions'])
mets_df = pd.DataFrame(dfs['reconstruction.xlsx:Metabolites'])

metabolites = mets_df['Metabolite ID'].tolist()
met_names = mets_df['Metabolite name'].tolist()

# Parse genes
genes = set()
gene_to_rxns = {}
for j, gpr in enumerate(rxns_df['GPR rule'].tolist()):
    if pd.isna(gpr):
        continue
    gene_ids = re.findall(r'MMSYN1_\d+', str(gpr))
    for g in gene_ids:
        genes.add(g)
        if g not in gene_to_rxns:
            gene_to_rxns[g] = []
        gene_to_rxns[g].append(j)

genes = sorted(list(genes))
gene_to_idx = {g: i for i, g in enumerate(genes)}

n_genes = len(genes)
n_mets = len(metabolites)
n_rxns = len(rxns_df)

# Build stoichiometry
S = np.zeros((n_mets, n_rxns))
for j, row in rxns_df.iterrows():
    equation = row['Reaction equation'].lower()
    for i, name in enumerate(met_names):
        if name.lower() in equation:
            left = equation.split('-->')[0] if '-->' in equation else equation.split('<==>')[0]
            S[i, j] = -1 if name.lower() in left else +1

# Gene-reaction matrix
G = np.zeros((n_genes, n_rxns))
for g, rxn_indices in gene_to_rxns.items():
    g_idx = gene_to_idx[g]
    for r_idx in rxn_indices:
        G[g_idx, r_idx] = 1

# Key metabolite indices
def find_met_idx(keyword):
    for i, n in enumerate(met_names):
        if keyword in n.lower():
            return i
    return 0

atp_idx = find_met_idx('atp')
adp_idx = find_met_idx('adp')
amp_idx = find_met_idx('amp')
glucose_idx = find_met_idx('d-glucose')
pyruvate_idx = find_met_idx('pyruvate')
lactate_idx = find_met_idx('lactate')

# Environment
ENV_VARS = ['glucose_ext', 'lactate_ext', 'oxygen', 'amino_acids', 'ph', 'temperature']
n_env = len(ENV_VARS)

# Analyze G matrix
genes_per_rxn = (G > 0).sum(axis=0)
rxns_per_gene = (G > 0).sum(axis=1)

print(f"Data: {n_genes} genes, {n_mets} metabolites, {n_rxns} reactions")
print(f"\nG matrix analysis:")
print(f"  Reactions with 0 genes (spontaneous): {(genes_per_rxn == 0).sum()}")
print(f"  Reactions with 1 gene: {(genes_per_rxn == 1).sum()}")
print(f"  Reactions with 2+ genes: {(genes_per_rxn >= 2).sum()}")
print(f"  Average reactions per gene: {rxns_per_gene.mean():.1f}")

In [ ]:
#@title 3️⃣ Trajectory Generator

class PhysicsTrajectoryGenerator:
    def __init__(self, S, G, n_genes, n_mets, n_rxns, device='cpu'):
        self.S = torch.tensor(S, dtype=torch.float32, device=device)
        self.G = torch.tensor(G, dtype=torch.float32, device=device)
        self.n_genes = n_genes
        self.n_mets = n_mets
        self.n_rxns = n_rxns
        self.device = device
        
        self.substrate_mask = (self.S < 0).float()
        self.Km = torch.rand(n_rxns, device=device) * 2.0 + 0.1
        self.Vmax_base = torch.rand(n_rxns, device=device) * 0.5 + 0.1
    
    def simulate(self, n_steps=50, dt=0.01, batch_size=1):
        gene_expr = torch.rand(batch_size, self.n_genes, device=self.device) * 1.5 + 0.5
        
        met_conc = torch.rand(batch_size, self.n_mets, device=self.device) * 1.0 + 0.5
        met_conc[:, atp_idx] = 4.0 + torch.rand(batch_size, device=self.device) * 0.5
        met_conc[:, adp_idx] = 0.5 + torch.rand(batch_size, device=self.device) * 0.2
        met_conc[:, glucose_idx] = 2.0 + torch.rand(batch_size, device=self.device) * 1.0
        met_conc[:, lactate_idx] = 0.1 + torch.rand(batch_size, device=self.device) * 0.1
        
        env = torch.zeros(batch_size, n_env, device=self.device)
        env[:, 0] = 10.0 + torch.rand(batch_size, device=self.device) * 5.0
        env[:, 2] = 1.0
        env[:, 3] = 5.0
        env[:, 4] = 7.0
        env[:, 5] = 1.0
        
        gene_traj = [gene_expr.clone()]
        met_traj = [met_conc.clone()]
        env_traj = [env.clone()]
        
        for step in range(n_steps):
            enzyme_activity = gene_expr @ self.G
            Vmax = enzyme_activity * self.Vmax_base.unsqueeze(0)
            
            n_substrates = self.substrate_mask.sum(dim=0).clamp(min=1)
            substrate_conc = (met_conc @ self.substrate_mask) / n_substrates + 0.001
            
            flux = Vmax * substrate_conc / (self.Km.unsqueeze(0) + substrate_conc)
            flux = flux.clamp(min=0)
            
            dM_dt = flux @ self.S.T
            
            glucose_ext = env[:, 0]
            glucose_uptake = 0.1 * glucose_ext / (0.5 + glucose_ext)
            glucose_uptake = glucose_uptake.clamp(max=glucose_ext * 0.1)
            
            lactate_int = met_conc[:, lactate_idx]
            lactate_export = 0.05 * lactate_int / (1.0 + lactate_int)
            
            met_conc = met_conc + dt * dM_dt
            glucose_add = torch.zeros_like(met_conc)
            glucose_add[:, glucose_idx] = glucose_uptake
            lactate_sub = torch.zeros_like(met_conc)
            lactate_sub[:, lactate_idx] = -lactate_export
            met_conc = (met_conc + glucose_add + lactate_sub).clamp(min=0.001)
            
            env_0_new = (env[:, 0] - glucose_uptake).clamp(min=0)
            env_1_new = env[:, 1] + lactate_export
            env_4_new = 7.0 - 0.1 * env_1_new.clamp(max=5.0)
            env = torch.stack([env_0_new, env_1_new, env[:, 2], env[:, 3], env_4_new, env[:, 5]], dim=-1)
            
            gene_expr = gene_expr + 0.001 * torch.randn_like(gene_expr)
            gene_expr = gene_expr.clamp(0.1, 2.0)
            
            gene_traj.append(gene_expr.clone())
            met_traj.append(met_conc.clone())
            env_traj.append(env.clone())
        
        return {
            'gene_trajectory': torch.stack(gene_traj, dim=1),
            'met_trajectory': torch.stack(met_traj, dim=1),
            'env_trajectory': torch.stack(env_traj, dim=1),
        }

generator = PhysicsTrajectoryGenerator(S, G, n_genes, n_mets, n_rxns, device)
print("✅ Generator ready")

In [ ]:
#@title 4️⃣ Dark Manifold V10 Model

class DarkManifoldV10(nn.Module):
    """
    V10: Biologically-Correct Knockouts
    
    Key change: Use G matrix DIRECTLY for flux gating.
    If a gene is required for a reaction, knockout STOPS that reaction.
    """
    
    def __init__(self, n_genes, n_mets, n_rxns, n_env, S, G, hidden_dim=256):
        super().__init__()
        
        self.n_genes = n_genes
        self.n_mets = n_mets
        self.n_rxns = n_rxns
        self.n_env = n_env
        
        # Fixed biochemistry
        self.register_buffer('S', torch.tensor(S, dtype=torch.float32))
        self.register_buffer('G', torch.tensor(G, dtype=torch.float32))
        self.register_buffer('substrate_mask', (torch.tensor(S) < 0).float())
        
        # Precompute: which reactions have genes vs spontaneous
        genes_per_rxn = self.G.sum(dim=0)  # (n_rxns,)
        self.register_buffer('has_genes', (genes_per_rxn > 0).float())
        self.register_buffer('genes_per_rxn', genes_per_rxn.clamp(min=1))  # Avoid div by 0
        
        # Gene regulation (W_reg) - keep from V9
        self.W_reg = nn.Parameter(torch.randn(n_genes, n_genes) * 0.1)
        
        # Environment sensing
        self.env_sensor = nn.Sequential(
            nn.Linear(n_env, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, n_genes),
            nn.Tanh(),
        )
        
        # Transport
        self.log_glucose_Km = nn.Parameter(torch.tensor(0.0))
        self.log_glucose_Vmax = nn.Parameter(torch.tensor(-1.0))
        self.log_lactate_Km = nn.Parameter(torch.tensor(0.0))
        self.log_lactate_Vmax = nn.Parameter(torch.tensor(-1.0))
        
        # Gene → Vmax (regulatory effects)
        self.gene_encoder = nn.Sequential(
            nn.Linear(n_genes, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_rxns),
            nn.Softplus(),
        )
        
        # Km values (learnable)
        self.log_Km = nn.Parameter(torch.randn(n_rxns) * 0.5)
        
        # Metabolite encoder
        self.met_encoder = nn.Sequential(
            nn.Linear(n_mets, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        
        # Hamiltonian
        self.hamiltonian_net = nn.Sequential(
            nn.Linear(n_mets + n_rxns, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 1),
        )
    
    @property
    def Km(self):
        return torch.exp(self.log_Km).clamp(0.01, 100.0)
    
    def compute_flux_gate(self, gene_expr):
        """
        V10 KEY FUNCTION: Biologically-correct flux gating.
        
        For each reaction:
        - If reaction has no required genes (spontaneous): gate = 1
        - If reaction has required genes: gate = PRODUCT of required gene expressions
        
        This means: if ANY required gene is 0, the reaction STOPS.
        """
        batch_size = gene_expr.shape[0]
        
        # Gene expression: (batch, n_genes)
        # G matrix: (n_genes, n_rxns)
        
        # For product, we use log-sum-exp trick:
        # product(x_i) = exp(sum(log(x_i)))
        
        # Clamp gene_expr to avoid log(0)
        gene_expr_safe = gene_expr.clamp(min=1e-6)
        
        # Log of gene expression: (batch, n_genes)
        log_expr = torch.log(gene_expr_safe)
        
        # Sum of logs for required genes: (batch, n_rxns)
        # Only sum where G[gene, rxn] = 1
        log_sum = log_expr @ self.G  # (batch, n_rxns)
        
        # Product of gene expressions for each reaction
        gene_product = torch.exp(log_sum)  # (batch, n_rxns)
        
        # For reactions with no genes, set gate = 1
        # For reactions with genes, use the product
        flux_gate = torch.where(
            self.has_genes.unsqueeze(0) > 0,
            gene_product,
            torch.ones_like(gene_product)
        )
        
        # Clamp to [0, 1] for stability
        flux_gate = flux_gate.clamp(0, 1)
        
        return flux_gate
    
    def forward(self, gene_expr, met_conc, env_state, dt=0.01):
        batch_size = gene_expr.shape[0]
        
        # 1. Gene regulation
        reg_signal = torch.tanh(gene_expr @ self.W_reg.T)
        env_signal = self.env_sensor(env_state)
        regulated_genes = gene_expr * (1.0 + 0.5 * reg_signal + 0.2 * env_signal)
        regulated_genes = regulated_genes.clamp(min=0.001)
        
        # 2. V10: Biologically-correct flux gate
        flux_gate = self.compute_flux_gate(regulated_genes)
        
        # 3. Vmax from gene encoder (captures regulatory complexity)
        Vmax = self.gene_encoder(regulated_genes)
        
        # 4. Michaelis-Menten kinetics
        n_substrates = self.substrate_mask.sum(dim=0).clamp(min=1)
        substrate_conc = (met_conc @ self.substrate_mask) / n_substrates + 0.001
        mm_term = substrate_conc / (self.Km.unsqueeze(0) + substrate_conc)
        
        # 5. Combined flux: Vmax × MM × GATE
        # The gate is the KEY - if required gene is 0, flux is 0
        flux = Vmax * mm_term * flux_gate
        
        # 6. Hamiltonian
        H = self.hamiltonian_net(torch.cat([met_conc, flux], dim=-1)).squeeze(-1)
        
        # 7. Stoichiometry
        dM_dt = flux @ self.S.T
        
        # 8. Transport
        glucose_ext = env_state[:, 0]
        glucose_Km = torch.exp(self.log_glucose_Km).clamp(0.01, 10.0)
        glucose_Vmax = torch.exp(self.log_glucose_Vmax).clamp(0.01, 2.0)
        glucose_uptake = glucose_Vmax * glucose_ext / (glucose_Km + glucose_ext)
        glucose_uptake = glucose_uptake.clamp(max=glucose_ext * 0.1)
        
        lactate_int = met_conc[:, lactate_idx]
        lactate_Km = torch.exp(self.log_lactate_Km).clamp(0.01, 10.0)
        lactate_Vmax = torch.exp(self.log_lactate_Vmax).clamp(0.01, 2.0)
        lactate_export = lactate_Vmax * lactate_int / (lactate_Km + lactate_int)
        lactate_export = lactate_export.clamp(max=lactate_int * 0.1)
        
        # 9. Update states
        met_next = met_conc + dt * dM_dt
        glucose_delta = torch.zeros_like(met_next)
        glucose_delta[:, glucose_idx] = glucose_uptake
        lactate_delta = torch.zeros_like(met_next)
        lactate_delta[:, lactate_idx] = -lactate_export
        met_next = (met_next + glucose_delta + lactate_delta).clamp(min=0.001)
        
        env_0 = (env_state[:, 0] - glucose_uptake).clamp(min=0)
        env_1 = env_state[:, 1] + lactate_export
        env_4 = 7.0 - 0.1 * env_1.clamp(max=5.0)
        env_next = torch.stack([env_0, env_1, env_state[:, 2], env_state[:, 3], env_4, env_state[:, 5]], dim=-1)
        
        return {
            'met_next': met_next,
            'env_next': env_next,
            'flux': flux,
            'flux_gate': flux_gate,
            'hamiltonian': H,
            'regulated_genes': regulated_genes,
            'dM_dt': dM_dt,
        }
    
    def rollout(self, gene_expr, met_init, env_init, n_steps, dt=0.01):
        met_traj = [met_init.clone()]
        env_traj = [env_init.clone()]
        flux_traj = []
        H_traj = []
        
        met = met_init.clone()
        env = env_init.clone()
        for _ in range(n_steps):
            out = self.forward(gene_expr, met, env, dt)
            met = out['met_next']
            env = out['env_next']
            met_traj.append(met)
            env_traj.append(env)
            flux_traj.append(out['flux'])
            H_traj.append(out['hamiltonian'])
        
        return {
            'met_trajectory': torch.stack(met_traj, dim=1),
            'env_trajectory': torch.stack(env_traj, dim=1),
            'flux_trajectory': torch.stack(flux_traj, dim=1),
            'hamiltonian_trajectory': torch.stack(H_traj, dim=1),
        }


model = DarkManifoldV10(
    n_genes=n_genes,
    n_mets=n_mets,
    n_rxns=n_rxns,
    n_env=n_env,
    S=S,
    G=G,
    hidden_dim=256,
).to(device)

print(f"DarkManifoldV10:")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Reactions with genes: {model.has_genes.sum().int().item()}")
print(f"  Spontaneous reactions: {(1 - model.has_genes).sum().int().item()}")

In [ ]:
#@title 5️⃣ Test Knockout Effect BEFORE Training

print("="*60)
print("V10 KNOCKOUT TEST (BEFORE TRAINING)")
print("="*60)

model.eval()

# Initial conditions
met_init = torch.ones(1, n_mets, device=device) * 0.5
met_init[0, atp_idx] = 4.0
met_init[0, adp_idx] = 0.5
met_init[0, glucose_idx] = 2.0

env_init = torch.zeros(1, n_env, device=device)
env_init[0, 0] = 10.0
env_init[0, 2] = 1.0
env_init[0, 3] = 5.0
env_init[0, 4] = 7.0
env_init[0, 5] = 1.0

# Wildtype
gene_wt = torch.ones(1, n_genes, device=device)
with torch.no_grad():
    wt_result = model.rollout(gene_wt, met_init.clone(), env_init.clone(), n_steps=50)
    wt_atp = wt_result['met_trajectory'][0, -1, atp_idx].item()
    
    # Check flux gate for wildtype
    flux_gate_wt = model.compute_flux_gate(gene_wt)
    print(f"\nWildtype:")
    print(f"  Final ATP: {wt_atp:.4f} mM")
    print(f"  Flux gate mean: {flux_gate_wt.mean().item():.4f}")
    print(f"  Flux gate min: {flux_gate_wt.min().item():.4f}")

# Test knockouts
ko_atp = []
ko_flux_gates = []
for i in tqdm(range(n_genes), desc="Testing knockouts"):
    gene_ko = torch.ones(1, n_genes, device=device)
    gene_ko[0, i] = 0.0
    
    with torch.no_grad():
        # Check flux gate
        flux_gate_ko = model.compute_flux_gate(gene_ko)
        ko_flux_gates.append(flux_gate_ko[0].cpu().numpy())
        
        # Rollout
        ko_result = model.rollout(gene_ko, met_init.clone(), env_init.clone(), n_steps=50)
        ko_atp.append(ko_result['met_trajectory'][0, -1, atp_idx].item())

ko_atp = np.array(ko_atp)
atp_ratios = ko_atp / (wt_atp + 1e-6)

print(f"\nKnockout results:")
print(f"  ATP range: [{ko_atp.min():.4f}, {ko_atp.max():.4f}] mM")
print(f"  ATP ratio range: [{atp_ratios.min():.4f}, {atp_ratios.max():.4f}]")
print(f"  ATP ratio std: {atp_ratios.std():.4f}")

# Check how many reactions are affected by each knockout
rxns_affected = []
for i, fg in enumerate(ko_flux_gates):
    # How many reactions dropped to 0?
    affected = (fg < 0.01).sum()
    rxns_affected.append(affected)

rxns_affected = np.array(rxns_affected)
print(f"\nReactions affected per knockout:")
print(f"  Mean: {rxns_affected.mean():.1f}")
print(f"  Max: {rxns_affected.max()}")
print(f"  Genes affecting 0 rxns: {(rxns_affected == 0).sum()}")
print(f"  Genes affecting 1+ rxns: {(rxns_affected > 0).sum()}")

# Essential genes
essential_mask = atp_ratios < 0.5
print(f"\nEssential genes (ATP < 50% WT): {essential_mask.sum()}/{n_genes}")

In [ ]:
#@title 6️⃣ Training

n_epochs = 500
batch_size = 8
n_steps = 30
lr = 3e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

losses = []

print(f"Training V10 for {n_epochs} epochs...")

for epoch in tqdm(range(n_epochs)):
    model.train()
    
    with torch.no_grad():
        target = generator.simulate(n_steps=n_steps, batch_size=batch_size)
    
    gene_expr = target['gene_trajectory'][:, 0].clone()
    met_init = target['met_trajectory'][:, 0].clone()
    env_init = target['env_trajectory'][:, 0].clone()
    true_met_traj = target['met_trajectory'].clone()
    
    pred = model.rollout(gene_expr, met_init.clone(), env_init.clone(), n_steps)
    
    # Loss
    loss_traj = F.mse_loss(pred['met_trajectory'], true_met_traj)
    loss_H = (pred['hamiltonian_trajectory'][:, 1:] - pred['hamiltonian_trajectory'][:, :-1]).pow(2).mean()
    
    # W_reg should be strong
    loss_wreg = F.relu(0.3 - model.W_reg.abs().max())
    
    total_loss = 10.0 * loss_traj + 1.0 * loss_H + 2.0 * loss_wreg
    
    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    
    losses.append(total_loss.item())
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}: Loss={total_loss.item():.4f}, Traj={loss_traj.item():.4f}")

print(f"\n✅ Training complete. Final loss: {losses[-1]:.4f}")

In [ ]:
#@title 7️⃣ Evaluate Trajectory

model.eval()

with torch.no_grad():
    test_target = generator.simulate(n_steps=50, batch_size=1)

gene_test = test_target['gene_trajectory'][:, 0]
met_test_init = test_target['met_trajectory'][:, 0]
env_test_init = test_target['env_trajectory'][:, 0]
true_traj = test_target['met_trajectory'][0].cpu().numpy()

with torch.no_grad():
    pred = model.rollout(gene_test, met_test_init.clone(), env_test_init.clone(), n_steps=50)

pred_traj = pred['met_trajectory'][0].cpu().numpy()

corr = np.corrcoef(true_traj.flatten(), pred_traj.flatten())[0, 1]
mse = np.mean((true_traj - pred_traj)**2)

print(f"Trajectory: Corr={corr:.4f}, MSE={mse:.6f}")

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
met_indices = [atp_idx, adp_idx, glucose_idx, lactate_idx, 0, 1]
met_labels = ['ATP', 'ADP', 'Glucose', 'Lactate', 'Met 0', 'Met 1']

for ax, idx, label in zip(axes.flatten(), met_indices, met_labels):
    ax.plot(true_traj[:, idx], 'b-', linewidth=2, label='True')
    ax.plot(pred_traj[:, idx], 'r--', linewidth=2, label='V10')
    ax.set_title(label)
    ax.legend()

plt.suptitle(f'V10 Trajectory (Corr={corr:.3f})', fontsize=14)
plt.tight_layout()
plt.savefig('trajectory_v10.png', dpi=150)
plt.show()

In [ ]:
#@title 8️⃣ Evaluate Knockouts (AFTER Training)

print("="*60)
print("V10 KNOCKOUT ANALYSIS (AFTER TRAINING)")
print("="*60)

model.eval()

# Initial conditions
met_init = torch.ones(1, n_mets, device=device) * 0.5
met_init[0, atp_idx] = 4.0
met_init[0, adp_idx] = 0.5
met_init[0, glucose_idx] = 2.0

env_init = torch.zeros(1, n_env, device=device)
env_init[0, 0] = 10.0
env_init[0, 2] = 1.0
env_init[0, 3] = 5.0
env_init[0, 4] = 7.0
env_init[0, 5] = 1.0

# Wildtype
gene_wt = torch.ones(1, n_genes, device=device)
with torch.no_grad():
    wt_result = model.rollout(gene_wt, met_init.clone(), env_init.clone(), n_steps=50)
    wt_atp = wt_result['met_trajectory'][0, -1, atp_idx].item()

print(f"\nWildtype ATP: {wt_atp:.4f} mM")

# All knockouts
ko_results = []
for i in tqdm(range(n_genes), desc="Knockouts"):
    gene_ko = torch.ones(1, n_genes, device=device)
    gene_ko[0, i] = 0.0
    
    with torch.no_grad():
        ko_result = model.rollout(gene_ko, met_init.clone(), env_init.clone(), n_steps=50)
        ko_atp = ko_result['met_trajectory'][0, -1, atp_idx].item()
        
        # Check flux gate
        flux_gate = model.compute_flux_gate(gene_ko)
        rxns_blocked = (flux_gate[0] < 0.01).sum().item()
    
    atp_ratio = ko_atp / (wt_atp + 1e-6)
    ko_results.append({
        'gene': genes[i],
        'atp': ko_atp,
        'atp_ratio': atp_ratio,
        'rxns_blocked': rxns_blocked,
        'is_essential': atp_ratio < 0.5,
    })

# Analysis
atp_ratios = np.array([r['atp_ratio'] for r in ko_results])
essential = [r for r in ko_results if r['is_essential']]
rxns_blocked = np.array([r['rxns_blocked'] for r in ko_results])

print(f"\nResults:")
print(f"  Essential: {len(essential)}/{n_genes} ({100*len(essential)/n_genes:.1f}%)")
print(f"  ATP ratio range: [{atp_ratios.min():.4f}, {atp_ratios.max():.4f}]")
print(f"  ATP ratio std: {atp_ratios.std():.4f}")
print(f"  ATP ratio variance: {atp_ratios.var():.4f}")

print(f"\nReactions blocked:")
print(f"  Mean: {rxns_blocked.mean():.1f}")
print(f"  Max: {rxns_blocked.max()}")

# Top essential
sorted_ko = sorted(ko_results, key=lambda x: x['atp_ratio'])
print(f"\nTop 10 most essential:")
for r in sorted_ko[:10]:
    status = "❌" if r['is_essential'] else "⚠️"
    print(f"  {status} {r['gene']}: ATP={r['atp_ratio']:.1%}, blocks {r['rxns_blocked']} rxns")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(atp_ratios, bins=30, edgecolor='black')
axes[0].axvline(x=0.5, color='r', linestyle='--', label='Lethal')
axes[0].set_xlabel('ATP ratio (KO/WT)')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Knockout ATP Distribution\n{len(essential)} essential')
axes[0].legend()

axes[1].scatter(rxns_blocked, atp_ratios, alpha=0.5)
axes[1].axhline(y=0.5, color='r', linestyle='--')
axes[1].set_xlabel('Reactions blocked')
axes[1].set_ylabel('ATP ratio')
axes[1].set_title('Blocked rxns vs ATP')

W_reg_np = model.W_reg.detach().cpu().numpy()
axes[2].imshow(np.abs(W_reg_np[:30, :30]), cmap='hot')
axes[2].set_title(f'|W_reg| (max={np.abs(W_reg_np).max():.2f})')

plt.tight_layout()
plt.savefig('knockout_v10.png', dpi=150)
plt.show()

In [ ]:
#@title 9️⃣ Save Model

Km_np = model.Km.detach().cpu().numpy()

save_dict = {
    'model_state_dict': model.state_dict(),
    'genes': genes,
    'metabolites': metabolites,
    'met_names': met_names,
    'env_vars': ENV_VARS,
    'stoichiometry': S,
    'gene_reaction_map': G,
    'training_losses': losses,
    'knockout_results': ko_results,
    'version': 'v10_biological_knockout',
    'features': [
        'biological_flux_gate',
        'G_matrix_direct',
        'product_knockout_logic',
    ],
    'metrics': {
        'trajectory_corr': float(corr),
        'trajectory_mse': float(mse),
        'n_essential': len(essential),
        'essential_pct': 100 * len(essential) / n_genes,
        'atp_variance': float(atp_ratios.var()),
        'atp_std': float(atp_ratios.std()),
        'wreg_max': float(np.abs(W_reg_np).max()),
        'km_std': float(Km_np.std()),
    }
}

torch.save(save_dict, 'dark_manifold_v10.pt')

print("="*60)
print("V10 SUMMARY")
print("="*60)
print(f"\nTrajectory: Corr={corr:.4f}")
print(f"Essential genes: {len(essential)}/{n_genes} ({100*len(essential)/n_genes:.1f}%)")
print(f"ATP variance: {atp_ratios.var():.4f}")
print(f"W_reg max: {np.abs(W_reg_np).max():.3f}")

print(f"\nComparison:")
print(f"          V8      V9      V10     Target")
print(f"Essential 0%      0%      {100*len(essential)/n_genes:.0f}%      50-70%")
print(f"ATP var   ~0      ~0      {atp_ratios.var():.3f}   >0.01")

from google.colab import files
files.download('dark_manifold_v10.pt')
files.download('trajectory_v10.png')
files.download('knockout_v10.png')